**Navigation** : [jumeau Python](ICT-49-JLensFideliteCrossTaille-Python.ipynb)

# ICT-49 — Stabilité cross-taille de la fidélité J-Lens

**Question.** Quand le modèle grossit de 1.7B à 27B, la lecture J-Lens reste-t-elle *stable* ? Concrètement : (1) les mêmes concepts sont-ils lus avec la même fidélité, et (2) la même structure de profondeur, à mesure que la taille augmente ?

Cet instrument répond en **CPU/C#** sur les cinq échelles déjà commitées. Il prolonge le tête-à-tête SAE↔J-Lens (ICT-24) et la géométrie factored (ICT-36) en ajoutant l'axe de la **taille du modèle**.

## Garde-fous d'honnêteté

- **Substrat commité** : les fichiers `traces/calib_jlens_*.npz` (1.7B, 2B, 8B, 9B, 27B), produits par `scripts/extract_jlens_fidelity.py` (#8236(c)). Aucune exécution GPU ici — on lit des résumés de calibration déjà calculés.
- **Provenance des lens** : 1.7B / 2B = lens **local-fit** ; 8B / 9B / 27B = lens **publiés** (neuronpedia). La comparaison cross-taille mélange donc deux provenances — à garder en tête.
- **Petit échantillon** : `n_eval` = 4 prompts/concept. Les chiffres sont une **lecture indicative**, pas une mesure statistique.
- **Profondeurs alignées** : chaque échelle calibre ses couches en profondeur **fractionnelle** 0.25 / 0.50 (ex. 1.7B → 7/14 sur 28 ; 27B → 16/32 sur 64). On compare donc à profondeur fractionnelle égale, pas à couche absolue.

In [1]:
using System;
using System.IO;
using System.IO.Compression;
using System.Text;
using System.Text.RegularExpressions;
using System.Globalization;
using System.Collections.Generic;
using System.Linq;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

// Chemins relatifs au carnet : --cwd = repertoire de la serie ICT-Series
string FindTraces()
{
    var dir = new DirectoryInfo(Directory.GetCurrentDirectory());
    while (dir != null)
    {
        foreach (var candidate in new[] { Path.Combine(dir.FullName, "traces"), Path.Combine(dir.FullName, "MyIA.AI.Notebooks", "IIT", "ICT-Series", "traces") })
            if (File.Exists(Path.Combine(candidate, "calib_jlens_qwen3-1-7b.npz"))) return candidate;
        dir = dir.Parent;
    }
    throw new DirectoryNotFoundException("Répertoire des traces introuvable depuis le dossier courant.");
}
string TRACES = FindTraces();
// 5 echelles, servables en CPU
var SCALES = new Dictionary<string, string> {
    ["1.7B"] = "calib_jlens_qwen3-1-7b.npz",
    ["2B"] = "calib_jlens_qwen3-5-2b.npz",
    ["8B"] = "calib_jlens_qwen3-8b.npz",
    ["9B"] = "calib_jlens_qwen3-5-9b.npz",
    ["27B"] = "calib_jlens_qwen3-5-27b.npz"
};
string[] ORDER = { "1.7B", "2B", "8B", "9B", "27B" };
string[] METRICS = { "overlap10", "rel_l2", "kl" };

// Lecture bornée aux tableaux numériques NPY 1.0, sans interprétation de pickle.
double[] ReadNpy(ZipArchiveEntry entry)
{
    using var stream = entry.Open();
    using var reader = new BinaryReader(stream);
    byte[] magic = reader.ReadBytes(6);
    if (!magic.SequenceEqual(new byte[] { 0x93, 0x4e, 0x55, 0x4d, 0x50, 0x59 })) throw new InvalidDataException("Signature NPY invalide : " + entry.FullName);
    byte major = reader.ReadByte(), minor = reader.ReadByte();
    if (major != 1 || minor != 0) throw new InvalidDataException("Version NPY non prise en charge.");
    int headerLength = reader.ReadUInt16();
    string header = Encoding.ASCII.GetString(reader.ReadBytes(headerLength));
    var descr = Regex.Match(header, "['\"]descr['\"]\\s*:\\s*['\"]([^'\"]+)['\"]");
    var shape = Regex.Match(header, "['\"]shape['\"]\\s*:\\s*\\(([^)]*)\\)");
    if (!descr.Success || !shape.Success || Regex.IsMatch(header, "['\"]fortran_order['\"]\\s*:\\s*True")) throw new InvalidDataException("En-tête NPY non pris en charge : " + header);
    string dtype = descr.Groups[1].Value;
    int count = 1;
    foreach (string part in shape.Groups[1].Value.Split(','))
        if (!string.IsNullOrWhiteSpace(part)) count = checked(count * int.Parse(part.Trim(), CultureInfo.InvariantCulture));
    if (count < 0 || count > 1000000) throw new InvalidDataException("Taille NPY invalide.");
    bool bigEndian = dtype.StartsWith(">");
    string kind = dtype.TrimStart('<', '>', '=', '|');
    int width = kind == "f8" || kind == "i8" || kind == "u8" ? 8 : kind == "f4" || kind == "i4" || kind == "u4" ? 4 : kind == "i2" || kind == "u2" ? 2 : kind == "i1" || kind == "u1" ? 1 : 0;
    if (width == 0) throw new InvalidDataException("Type NPY non numérique : " + dtype);
    var result = new double[count];
    for (int i = 0; i < count; i++)
    {
        byte[] bytes = reader.ReadBytes(width);
        if (bytes.Length != width) throw new EndOfStreamException(entry.FullName);
        if (width > 1 && bigEndian == BitConverter.IsLittleEndian) Array.Reverse(bytes);
        switch (kind)
        {
            case "f8": result[i] = BitConverter.ToDouble(bytes, 0); break;
            case "f4": result[i] = BitConverter.ToSingle(bytes, 0); break;
            case "i8": result[i] = BitConverter.ToInt64(bytes, 0); break;
            case "u8": result[i] = BitConverter.ToUInt64(bytes, 0); break;
            case "i4": result[i] = BitConverter.ToInt32(bytes, 0); break;
            case "u4": result[i] = BitConverter.ToUInt32(bytes, 0); break;
            case "i2": result[i] = BitConverter.ToInt16(bytes, 0); break;
            case "u2": result[i] = BitConverter.ToUInt16(bytes, 0); break;
            case "i1": result[i] = unchecked((sbyte)bytes[0]); break;
            case "u1": result[i] = bytes[0]; break;
        }
    }
    return result;
}

var rows = new List<(string scale, string concept, int layer, double frac_depth, double overlap10, double rel_l2, double kl)>();
foreach (string scale in ORDER)
{
    using var archive = ZipFile.OpenRead(Path.Combine(TRACES, SCALES[scale]));
    var layerEntry = archive.GetEntry("meta_layers.npy") ?? throw new InvalidDataException("meta_layers absent");
    var layers = ReadNpy(layerEntry);
    if (layers.Length != 2) throw new InvalidDataException("Deux couches attendues.");
    double total = 2 * layers[layers.Length - 1]; // 2e couche en profondeur 0.50 -> total de couches
    foreach (var entry in archive.Entries)
    {
        string key = entry.FullName.EndsWith(".npy", StringComparison.Ordinal) ? entry.FullName.Substring(0, entry.FullName.Length - 4) : entry.FullName;
        if (key.StartsWith("meta", StringComparison.Ordinal)) continue;
        int separator = key.IndexOf("__", StringComparison.Ordinal);
        if (separator < 0) throw new InvalidDataException("Clé inattendue : " + key);
        string concept = key.Substring(0, separator);
        int ly = int.Parse(key.Substring(separator + 2), CultureInfo.InvariantCulture);
        var v = ReadNpy(entry); // [overlap10, rel_l2, kl]
        if (v.Length != 3 || v.Any(x => double.IsNaN(x) || double.IsInfinity(x))) throw new InvalidDataException("Métriques invalides : " + key);
        rows.Add((scale, concept, ly, Math.Round(ly / total, 4), v[0], v[1], v[2]));
    }
}
var concepts = rows.Select(r => r.concept).Distinct().OrderBy(x => x, StringComparer.Ordinal).ToArray();
var depths = rows.Select(r => r.frac_depth).Distinct().OrderBy(x => x).ToArray();
if (rows.Count != 50 || rows.Select(r => r.scale).Distinct().Count() != 5 || concepts.Length != 5 || !depths.SequenceEqual(new[] { 0.25, 0.5 })) throw new InvalidDataException("Structure de la table longue différente de la référence citée.");
if (rows.GroupBy(r => (r.scale, r.concept, r.frac_depth)).Any(g => g.Count() != 1)) throw new InvalidDataException("Observation dupliquée.");
Console.WriteLine($"Table longue : {rows.Count} lignes, {rows.Select(r => r.scale).Distinct().Count()} echelles, {concepts.Length} concepts");
Console.WriteLine("Profondeurs fractionnelles presentes : [" + string.Join(", ", depths.Select(x => x.ToString("0.##", CultureInfo.InvariantCulture))) + "]");

double Metric((string scale, string concept, int layer, double frac_depth, double overlap10, double rel_l2, double kl) row, string metric)
{
    switch (metric) { case "overlap10": return row.overlap10; case "rel_l2": return row.rel_l2; case "kl": return row.kl; default: throw new ArgumentException("Métrique inconnue : " + metric); }
}

Table longue : 50 lignes, 5 echelles, 5 concepts


Profondeurs fractionnelles presentes : [0.25, 0.5]


## 1. Tendance de fidélité par échelle

Trois métriques par (concept, couche) : `overlap@10` (plus **haut** = plus fidèle), `rel_l2` et `kl` (plus **bas** = plus fidèle). On les moyenne d'abord sur les 5 concepts × 2 profondeurs, par échelle.

In [2]:
// Références citées : sortie Python arrondie à six décimales.
// La demi-unité du dernier chiffre affiché borne l'écart à la valeur non arrondie.
const double tableTolerance = 0.00000051;
double[,] referenceByScale = {
    { 0.013776, 0.767758, 17.496718 },
    { 0.028427, 1.230310, 9.628378 },
    { 0.008420, 1.590495, 18.909991 },
    { 0.026780, 0.964609, 11.900528 },
    { 0.018645, 1.114448, 11.175254 }
};
var by_scale = new Dictionary<string, Dictionary<string, double>>();
double maxScaleError = 0;
Console.WriteLine("Fidelite moyenne par echelle (overlap10 : plus haut = plus fidele ; rel_l2/kl : plus bas = plus fidele)");
Console.WriteLine("       overlap10    rel_l2         kl");
Console.WriteLine("scale");
for (int i = 0; i < ORDER.Length; i++)
{
    string scale = ORDER[i];
    var group = rows.Where(r => r.scale == scale).ToArray();
    if (group.Length != 10) throw new InvalidDataException("Effectif incorrect pour " + scale);
    by_scale[scale] = new Dictionary<string, double>();
    for (int j = 0; j < METRICS.Length; j++)
    {
        double value = group.Average(r => Metric(r, METRICS[j]));
        by_scale[scale][METRICS[j]] = value;
        double error = Math.Abs(value - referenceByScale[i, j]);
        maxScaleError = Math.Max(maxScaleError, error);
        if (error > tableTolerance) throw new InvalidDataException(FormattableString.Invariant($"Moyenne {scale}/{METRICS[j]} : écart {error:G17} à la référence citée."));
    }
    Console.WriteLine(FormattableString.Invariant($"{scale,-5} {by_scale[scale]["overlap10"],10:F6} {by_scale[scale]["rel_l2"],9:F6} {by_scale[scale]["kl"],10:F6}"));
}
Console.WriteLine(FormattableString.Invariant($"Écart maximal à la table Python citée : {maxScaleError:G6} (tolérance {tableTolerance:G6})."));

Fidelite moyenne par echelle (overlap10 : plus haut = plus fidele ; rel_l2/kl : plus bas = plus fidele)


       overlap10    rel_l2         kl


scale


1.7B    0.013776  0.767758  17.496718


2B      0.028427  1.230310   9.628378


8B      0.008420  1.590495  18.909991


9B      0.026780  0.964609  11.900528


27B     0.018645  1.114448  11.175254


Écart maximal à la table Python citée : 4.99313E-07 (tolérance 5.1E-07).


**Lecture du résultat.** La fidélité moyenne n'évolue pas de façon monotone avec la taille : `overlap10` est le plus haut à 2B (`0.028427`) et le plus bas à 8B (`0.008420`), tandis que `rel_l2` croît de `0.767758` (1.7B) à `1.590495` (8B) avant de redescendre à `0.964609` (9B). L'écart maximal à la table Python citée, `4.99313E-07`, reste sous la tolérance de `5.1E-07`.

In [3]:
// Référence citée : tableau Python affiché à six décimales.
double[,] referencePivot = {
    { 0.920388, 1.299443, 1.620747, 1.048033, 1.287340 },
    { 0.760831, 1.269871, 1.620004, 0.932116, 1.104762 },
    { 0.766815, 1.276766, 1.582112, 0.952915, 1.124314 },
    { 0.518270, 0.962792, 1.512703, 0.877358, 0.970806 },
    { 0.872486, 1.342679, 1.616907, 1.012625, 1.085015 }
};
string[] referenceConcepts = { "code_python", "dialogue", "math", "narrative_en", "prose_fr" };
if (!concepts.SequenceEqual(referenceConcepts)) throw new InvalidDataException("Concepts différents de la référence citée.");
var pv = new Dictionary<string, Dictionary<string, double>>();
double maxPivotError = 0;
Console.WriteLine("rel_l2 moyen par concept et echelle (plus bas = plus fidele)");
Console.WriteLine("scale             1.7B        2B        8B        9B       27B");
Console.WriteLine("concept");
for (int i = 0; i < concepts.Length; i++)
{
    string concept = concepts[i];
    pv[concept] = new Dictionary<string, double>();
    var line = new StringBuilder();
    line.Append(FormattableString.Invariant($"{concept,-12}"));
    for (int j = 0; j < ORDER.Length; j++)
    {
        var group = rows.Where(r => r.concept == concept && r.scale == ORDER[j]).ToArray();
        if (group.Length != 2) throw new InvalidDataException("Deux profondeurs attendues.");
        double value = group.Average(r => r.rel_l2);
        pv[concept][ORDER[j]] = value;
        double error = Math.Abs(value - referencePivot[i, j]);
        maxPivotError = Math.Max(maxPivotError, error);
        if (error > tableTolerance) throw new InvalidDataException(FormattableString.Invariant($"Pivot {concept}/{ORDER[j]} : écart {error:G17} à la référence citée."));
        line.Append(FormattableString.Invariant($" {value,9:F6}"));
    }
    Console.WriteLine(line.ToString());
}
Console.WriteLine(FormattableString.Invariant($"Écart maximal à la table Python citée : {maxPivotError:G6} (tolérance {tableTolerance:G6})."));

rel_l2 moyen par concept et echelle (plus bas = plus fidele)


scale             1.7B        2B        8B        9B       27B


concept


code_python   0.920388  1.299443  1.620747  1.048033  1.287340


dialogue      0.760831  1.269871  1.620004  0.932116  1.104762


math          0.766815  1.276766  1.582112  0.952915  1.124314


narrative_en  0.518270  0.962792  1.512703  0.877358  0.970806


prose_fr      0.872486  1.342679  1.616907  1.012625  1.085015


Écart maximal à la table Python citée : 4.71163E-07 (tolérance 5.1E-07).


## 2. Stabilité du classement cross-taille

La question centrale : **les mêmes concepts sont-ils classés de la même façon d'une échelle à l'autre ?** Pour chaque métrique, on range les 5 concepts *dans* chaque échelle, puis on mesure la corrélation de rang de **Spearman** entre toutes les paires d'échelles. L'indice de stabilité est la moyenne de ces corrélations (1 = classement identique partout, 0 = aucun accord).

In [4]:
// Spearman : Pearson appliqué aux rangs moyens en cas d'ex æquo.
double stability_index(string metric, bool ascending)
{
    var ranks = new Dictionary<string, Dictionary<string, double>>();
    foreach (string scale in ORDER)
    {
        var means = rows.Where(r => r.scale == scale)
            .GroupBy(r => r.concept)
            .Select(g => (concept: g.Key, value: g.Average(r => Metric(r, metric))))
            .ToArray();
        var sorted = ascending ? means.OrderBy(x => x.value).ToArray() : means.OrderByDescending(x => x.value).ToArray();
        var rank = new Dictionary<string, double>();
        int start = 0;
        while (start < sorted.Length)
        {
            int end = start + 1;
            while (end < sorted.Length && sorted[end].value == sorted[start].value) end++;
            double averageRank = ((start + 1) + end) / 2.0;
            for (int k = start; k < end; k++) rank[sorted[k].concept] = averageRank;
            start = end;
        }
        ranks[scale] = rank;
    }
    var rhos = new List<double>();
    for (int i = 0; i < ORDER.Length; i++)
        for (int j = i + 1; j < ORDER.Length; j++)
        {
            var a = ranks[ORDER[i]];
            var b = ranks[ORDER[j]];
            var common = a.Keys.Intersect(b.Keys).OrderBy(x => x, StringComparer.Ordinal).ToArray();
            if (common.Length != concepts.Length) throw new InvalidDataException("Concepts non alignés.");
            double meanA = common.Average(c => a[c]);
            double meanB = common.Average(c => b[c]);
            double covariance = common.Sum(c => (a[c] - meanA) * (b[c] - meanB));
            double varianceA = common.Sum(c => (a[c] - meanA) * (a[c] - meanA));
            double varianceB = common.Sum(c => (b[c] - meanB) * (b[c] - meanB));
            if (varianceA == 0 || varianceB == 0) throw new InvalidDataException("Corrélation de rang indéfinie.");
            rhos.Add(covariance / Math.Sqrt(varianceA * varianceB));
        }
    if (rhos.Count != 10) throw new InvalidDataException("Nombre de paires incorrect.");
    return rhos.Average();
}

// sens : overlap10 plus haut = plus fidele (descending) ; rel_l2/kl plus bas = plus fidele (ascending)
var stab = new Dictionary<string, double> {
    ["overlap10"] = stability_index("overlap10", false),
    ["rel_l2"] = stability_index("rel_l2", true),
    ["kl"] = stability_index("kl", true)
};
// Références Python citées à trois décimales : demi-unité d'arrondi.
const double stabilityTolerance = 0.00050001;
var referenceStability = new Dictionary<string, double> { ["overlap10"] = 0.730, ["rel_l2"] = 0.720, ["kl"] = 0.750 };
double maxStabilityError = 0;
Console.WriteLine("Indice de stabilite du classement de concepts (1 = identique dans toutes les echelles)");
foreach (string metric in METRICS)
{
    double value = stab[metric];
    double error = Math.Abs(value - referenceStability[metric]);
    maxStabilityError = Math.Max(maxStabilityError, error);
    if (error > stabilityTolerance) throw new InvalidDataException(FormattableString.Invariant($"Stabilité {metric} : écart {error:G17} à la référence citée."));
    Console.WriteLine($"  {metric,-10} : {value.ToString("+0.000;-0.000;+0.000", CultureInfo.InvariantCulture)}");
}
Console.WriteLine(FormattableString.Invariant($"Écart maximal aux indices Python cités : {maxStabilityError:G6} (tolérance {stabilityTolerance:G6})."));

Indice de stabilite du classement de concepts (1 = identique dans toutes les echelles)


  overlap10  : +0.730


  rel_l2     : +0.720


  kl         : +0.750


Écart maximal aux indices Python cités : 1.11022E-16 (tolérance 0.00050001).


**Lecture du résultat.** Les trois indices de stabilité sont proches et positifs : `+0.730` (`overlap10`), `+0.720` (`rel_l2`) et `+0.750` (`kl`). Le classement des concepts se conserve donc en grande partie d'une échelle à l'autre, sans être identique.

In [5]:
// Dans chaque echelle, l'ecart entre la couche profonde (0.50) et la superficielle (0.25)
// Références citées : sortie Python arrondie à six décimales.
double[,] referenceDelta = {
    { 0.015004,  0.006742, -9.961708 },
    { 0.027682, -0.010502, -2.889779 },
    { 0.005486,  0.015486, -4.075736 },
    { 0.033636, -0.013428, -5.091580 },
    { 0.013844, -0.306271, -1.531882 }
};
var delta = new Dictionary<string, Dictionary<string, double>>();
double maxDeltaError = 0;
Console.WriteLine("Delta (profond 0.50  minus  superficiel 0.25) par echelle. Le SIGNE est-il stable cross-taille ?");
Console.WriteLine("       overlap10    rel_l2        kl");
Console.WriteLine("scale");
for (int i = 0; i < ORDER.Length; i++)
{
    string scale = ORDER[i];
    var deep = rows.Where(r => r.scale == scale && r.frac_depth == 0.50).ToArray();
    var shallow = rows.Where(r => r.scale == scale && r.frac_depth == 0.25).ToArray();
    if (deep.Length != 5 || shallow.Length != 5) throw new InvalidDataException("Effectifs par profondeur incorrects : " + scale);
    delta[scale] = new Dictionary<string, double>();
    for (int j = 0; j < METRICS.Length; j++)
    {
        string metric = METRICS[j];
        double value = deep.Average(r => Metric(r, metric)) - shallow.Average(r => Metric(r, metric));
        delta[scale][metric] = value;
        double error = Math.Abs(value - referenceDelta[i, j]);
        maxDeltaError = Math.Max(maxDeltaError, error);
        if (error > tableTolerance) throw new InvalidDataException(FormattableString.Invariant($"Delta {scale}/{metric} : écart {error:G17} à la référence citée."));
    }
    Console.WriteLine(FormattableString.Invariant($"{scale,-5} {delta[scale]["overlap10"],10:F6} {delta[scale]["rel_l2"],9:F6} {delta[scale]["kl"],10:F6}"));
}
Console.WriteLine(FormattableString.Invariant($"Écart maximal à la table Python citée : {maxDeltaError:G6} (tolérance {tableTolerance:G6})."));

Delta (profond 0.50  minus  superficiel 0.25) par echelle. Le SIGNE est-il stable cross-taille ?


       overlap10    rel_l2        kl


scale


1.7B    0.015004  0.006742  -9.961708


2B      0.027682 -0.010502  -2.889779


8B      0.005486  0.015486  -4.075736


9B      0.033636 -0.013428  -5.091580


27B     0.013844 -0.306271  -1.531882


Écart maximal à la table Python citée : 4.84412E-07 (tolérance 5.1E-07).


## 3. Lecture

Ce qui compte ici n'est pas la valeur absolue de la fidélité, mais **ce qui bouge et ce qui ne bouge pas** quand la taille change. Trois faits mesurés, repris en une vue d'ensemble :

In [6]:
// resume des 3 faits mesurés
// Références citées : tableaux Python affichés à quatre et trois décimales.
const double summaryTolerance = 0.00005001;
double[,] referenceSummary = {
    { 0.0138, 0.7678, 17.4967 },
    { 0.0284, 1.2303, 9.6284 },
    { 0.0084, 1.5905, 18.9100 },
    { 0.0268, 0.9646, 11.9005 },
    { 0.0186, 1.1144, 11.1753 }
};
int[,] referenceSigns = {
    { 1, 1, -1 }, { 1, -1, -1 }, { 1, 1, -1 },
    { 1, -1, -1 }, { 1, -1, -1 }
};
double maxSummaryError = 0;
Console.WriteLine("=== 1) Tendance de la fidelite (moyenne par echelle) ===");
Console.WriteLine("       overlap10  rel_l2       kl");
Console.WriteLine("scale");
for (int i = 0; i < ORDER.Length; i++)
{
    string scale = ORDER[i];
    for (int j = 0; j < METRICS.Length; j++)
    {
        double error = Math.Abs(by_scale[scale][METRICS[j]] - referenceSummary[i, j]);
        maxSummaryError = Math.Max(maxSummaryError, error);
        if (error > summaryTolerance) throw new InvalidDataException(FormattableString.Invariant($"Résumé {scale}/{METRICS[j]} : écart {error:G17} à la référence citée."));
    }
    Console.WriteLine(FormattableString.Invariant($"{scale,-5} {by_scale[scale]["overlap10"],10:F4} {by_scale[scale]["rel_l2"],7:F4} {by_scale[scale]["kl"],8:F4}"));
}
Console.WriteLine("\n=== 2) Stabilite du classement de concepts (Spearman moyen) ===");
foreach (string metric in METRICS)
{
    double value = stab[metric];
    if (Math.Abs(value - referenceStability[metric]) > stabilityTolerance) throw new InvalidDataException("Indice différent de la référence citée : " + metric);
    Console.WriteLine($"  {metric,-10} : {value.ToString("+0.000;-0.000;+0.000", CultureInfo.InvariantCulture)}");
}
Console.WriteLine("\n=== 3) Signe de l'ecart profondeur (profond - superficiel) par echelle ===");
Console.WriteLine("       overlap10  rel_l2   kl");
Console.WriteLine("scale");
int maxSignError = 0;
for (int i = 0; i < ORDER.Length; i++)
{
    string scale = ORDER[i];
    int[] signs = METRICS.Select(metric => Math.Sign(delta[scale][metric])).ToArray();
    for (int j = 0; j < METRICS.Length; j++)
    {
        int error = Math.Abs(signs[j] - referenceSigns[i, j]);
        maxSignError = Math.Max(maxSignError, error);
        if (error != 0) throw new InvalidDataException($"Signature différente de la référence citée : {scale}/{METRICS[j]}.");
    }
    Console.WriteLine(FormattableString.Invariant($"{scale,-5} {(double)signs[0],10:F1} {(double)signs[1],7:F1} {(double)signs[2],4:F1}"));
}
// Les lectures qualitatives sont vérifiées sur les valeurs calculées.
foreach (string scale in ORDER)
{
    var sorted = concepts.OrderBy(c => pv[c][scale]).ToArray();
    if (sorted[0] != "narrative_en" || !sorted.Skip(3).Contains("code_python")) throw new InvalidDataException("Classement qualitatif différent : " + scale);
}
if (ORDER.Any(s => delta[s]["overlap10"] <= 0 || delta[s]["kl"] >= 0) || !ORDER.Any(s => delta[s]["rel_l2"] > 0) || !ORDER.Any(s => delta[s]["rel_l2"] < 0)) throw new InvalidDataException("Signature qualitative différente.");
foreach (string metric in METRICS)
{
    var differences = Enumerable.Range(1, ORDER.Length - 1).Select(i => by_scale[ORDER[i]][metric] - by_scale[ORDER[i - 1]][metric]).ToArray();
    if (!differences.Any(x => x > 0) || !differences.Any(x => x < 0)) throw new InvalidDataException("Tendance monotone inattendue : " + metric);
}
Console.WriteLine("\nBilan de parité — écarts maximaux aux références Python citées :");
Console.WriteLine(FormattableString.Invariant($"  moyennes : {maxScaleError:G6} ; tableau croisé : {maxPivotError:G6} ; stabilité : {maxStabilityError:G6}"));
Console.WriteLine(FormattableString.Invariant($"  écarts de profondeur : {maxDeltaError:G6} ; résumé : {maxSummaryError:G6} ; signature : {maxSignError}"));

=== 1) Tendance de la fidelite (moyenne par echelle) ===


       overlap10  rel_l2       kl


scale


1.7B      0.0138  0.7678  17.4967


2B        0.0284  1.2303   9.6284


8B        0.0084  1.5905  18.9100


... (13 outputs left out)

27B          1.0    -1.0 -1.0



Bilan de parité — écarts maximaux aux références Python citées :


  moyennes : 4.99313E-07 ; tableau croisé : 4.71163E-07 ; stabilité : 1.11022E-16


  écarts de profondeur : 4.84412E-07 ; résumé : 4.75728E-05 ; signature : 0


## 3 (suite) — Lecture

Trois faits mesurés, sur les cinq échelles CPU :

1. **Le classement des concepts est substantiellement stable cross-taille.** L'indice de Spearman (classement des 5 concepts, moyen sur les paires d'échelles) tient à **0.72 – 0.75** pour les trois métriques — un accord fort, sans être parfait. Concrètement, sur `rel_l2` (plus bas = plus fidèle), `narrative_en` est le concept **le plus fidèle dans chacune des cinq échelles** et `code_python` le **moins fidèle sur quatre échelles sur cinq** — à 2B, c'est `prose_fr` qui ferme la marche. Entre ces deux extrémités, l'ordre bouge d'une échelle à l'autre : `prose_fr` est **4ᵉ à 1.7B, 5ᵉ à 2B, 3ᵉ à 8B, 4ᵉ à 9B, puis 2ᵉ à 27B**. Ce qui est lu de façon stable, c'est donc **les deux bouts du classement**, pas *à quel niveau*.

2. **La structure de profondeur est stable sur deux métriques sur trois.** Le signe de l'écart (couche 0.50 − couche 0.25) est **constant** pour `overlap10` (toujours + : la couche profonde capture plus de top-10) et pour `kl` (toujours − : la couche profonde est plus proche du modèle) — sur ces deux métriques et sur cet échantillon, la lecture J-Lens s'améliore en profondeur. Pour `rel_l2`, le signe **s'inverse** d'une échelle à l'autre : pas de direction stable.

3. **La magnitude de la fidélité n'est pas monotone en taille.** Ni `overlap10`, ni `rel_l2`, ni `kl` ne suit une tendance nette de 1.7B à 27B : elles oscillent. **C'est la réponse honnête au Concern 14B/27B du 16/11** : sur cet axe de taille, la fidélité ne « s'améliore pas » de façon prévisible avec la croissance du modèle — une divergence 14B/27B n'est donc **pas surprenante** au vu de ces cinq points.

**Limiter la portée.** Deux provenances de lens (1.7B/2B local-fit, 8B/9B/27B publiés) et `n_eval` = 4 : ce que ce carnet établit est une **observation sur ces cinq traces** : l'indice de stabilité repose sur le *classement* (peu sensible au bruit), le signe de profondeur sur une *différence de moyennes* (plus sensible) — **aucun test de sensibilité n'étant montré ici, ni l'un ni l'autre n'est encore une robustesse testée** ; les **niveaux** absolus de fidélité restent indicatifs. L'extension 14B/27B sur le même instrument (#27038) dira si l'accord 0.73 se maintient ou se dégrade franchement au-delà de 9B.

## Exercices

Trois prolongements de l'instrument. Chaque cellule code est un stub à compléter (pas d'erreur volontaire : le notebook s'exécute de bout en bout).

### Exercice 1 — Fidélité d'une échelle

Calculer la moyenne de `overlap10` pour une échelle donnée (argument `scale`).

In [7]:
// Moyenne de overlap10 sur tous les concepts et les 2 profondeurs d'une echelle.
// TODO etudiant : filtrer df sur scale, puis moyenner la colonne 'overlap10'.
double overlap_moyen(string scale)
{
    double result = 0.0; // TODO etudiant
    return result;
}

### Exercice 2 — Classer les échelles pour un concept

Pour un concept donné, retourner les échelles triées de la **plus fidèle** à la **moins fidèle** selon `rel_l2` (le plus bas d'abord).

In [8]:
// Echelles triees, du rel_l2 le plus bas (fidele) au plus haut, pour un concept.
// TODO etudiant : filtrer df sur concept, moyenner rel_l2 par scale, trier.
List<string> echelles_par_fidelite(string concept)
{
    var result = new List<string>(); // TODO etudiant
    return result;
}

### Exercice 3 — Stabilité d'une métrique

Réutiliser la logique de `stability_index` pour retourner l'indice de stabilité d'une métrique donnée, avec le sens de tri approprié.

In [9]:
// Indice de stabilite (Spearman moyen) pour une metrique, sens adapte.
// Indice : overlap10 est trié descending, rel_l2/kl sont triés ascending.
// TODO etudiant : determiner le sens puis appeler stability_index.
double indice_stabilite(string metric)
{
    double result = 0.0; // TODO etudiant
    return result;
}

## Conclusion

Les traces fournissent un instrument reproductible pour comparer la fidélité J-Lens et sa structure en profondeur à cinq tailles de modèle. Sur cet échantillon, le classement des concepts reste substantiellement stable, tandis que les niveaux de fidélité ne progressent pas de façon monotone avec la taille. Ces observations restent limitées par les deux provenances des lens et le petit nombre de prompts évalués.

L'extension à 14B/27B permettra de poursuivre la comparaison sur le même instrument. Une autre perspective est d'étudier la dynamique sous perturbation, afin de tester si les classements et les signatures de profondeur observés ici se maintiennent lorsque les conditions changent.

## À retenir

- **La question se mesure.** « La lecture J-Lens est-elle stable quand le modèle grossit ? » devient **des mesures reproductibles** sur les cinq échelles CPU déjà commitées : la fidélité de lecture et la structure de profondeur, suivies d'une échelle en taille.
- **Le classement tient, la magnitude non.** Quand la taille monte, le **classement des concepts** lu par J-Lens est stable (Spearman ~0,72–0,75) — mais la **magnitude** de la fidélité n'est **pas monotone** en taille. Stabilité de l'ordre ≠ croissance de la précision.
- **Le sens de tri lit le classement, pas l'indice.** `overlap10` est triée **décroissante** (plus haut = plus fidèle), `rel_l2` et `kl` **croissantes** (plus bas = plus fidèle) : c'est ce sens qui dit *quels* concepts sont lus comme les plus fidèles ; pris à l'envers, le même tableau dirait l'inverse. L'indice de stabilité, lui, ne dépend pas de ce choix : inverser l'ordre **dans toutes les échelles** retourne les rangs des deux côtés de chaque paire et laisse chaque corrélation de Spearman inchangée (`rel_l2` reste à +0.720) ; il ne changerait que si l'orientation changeait **d'une échelle à l'autre**.

## Vérifiez votre compréhension

1. `overlap10` est triée **décroissante** mais `rel_l2`/`kl` **croissantes** : que doit faire `indice_stabilite` pour que l'indice de Spearman moyen ait un sens sur les deux familles de métriques ?
2. Que distingue ce carnet entre « le **classement** des concepts est stable » et « la **magnitude** de la fidélité croît » ? Pourquoi l'une des deux est-elle le résultat porteur ?
3. Sur quelles **cinq échelles** CPU l'instrument est-il calibré, et quelle extension en porte la suite en taille ?

## Pour aller plus loin

- **La suite en taille** — 14B/27B sur le **même instrument** — est portée par l'extension GPU ([#27038](https://github.com/jsboige/CoursIA/issues/27038)).
- **Les carnets frères** : le tête-à-tête SAE↔J-Lens (ICT-24) et la géométrie factored (ICT-36) — ce carnet leur ajoute l'axe de la **taille du modèle**.